<a href="https://colab.research.google.com/github/genaiconference/Agentic_KAG_Workshop_DHS_2026/blob/main/02_knowledge_graph_creation_with_simplekgpipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Building a Knowledge Graph from an Unstructured PDF with Neo4j SimpleKGPipeline

This notebook demonstrates how to turn an **unstructured PDF document** — `Leave_Policy.pdf` — into a **knowledge graph** using Neo4j GraphRAG's [`SimpleKGPipeline`](https://neo4j.com/docs/neo4j-graphrag-python/current/user_guide_kg_builder.html).

The pipeline automatically produces two complementary graphs:

| Graph | What it captures | Node labels |
|-------|------------------|-------------|
| **Lexical graph** | The physical structure of the text | `Document`, `Chunk` |
| **Domain graph** | The *meaning* extracted from the text | `Policy`, `LeaveType`, `Employee`, `Eligibility`, ... |

**Workflow**
1. Load the PDF and split it into chunks
2. Embed each chunk (lexical graph)
3. Use an LLM to extract entities & relationships guided by a schema (domain graph)
4. Write everything to Neo4j and connect chunks to the entities they mention
5. Validate and query the result with Cypher

> This showcases the shift from *document-centric* retrieval to *graph-powered reasoning* over unstructured HR policy content.

In [1]:
!git clone https://github.com/genaiconference/Agentic_KAG_Workshop_DHS_2026.git

Cloning into 'Agentic_KAG_Workshop_DHS_2026'...
remote: Enumerating objects: 145, done.
remote: Counting objects: 100% (51/51), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 145 (delta 47), reused 41 (delta 41), pack-reused 94 (from 1)
Receiving objects: 100% (145/145), 3.56 MiB | 21.32 MiB/s, done.
Resolving deltas: 100% (76/76), done.


## 1. Install Required Packages

We install `neo4j-graphrag` (with the OpenAI extra), the Neo4j Python driver, and PDF handling dependencies. `SimpleKGPipeline` uses `pypdf` under the hood to read PDF files when `from_pdf=True`.

In [1]:
%pip install -U "neo4j-graphrag[openai]" neo4j pypdf python-dotenv pandas --quiet

## 2. Import Required Libraries

We import the Neo4j driver plus the GraphRAG building blocks: `SimpleKGPipeline`, an `OpenAILLM` for extraction and `OpenAIEmbeddings` for the lexical graph vectors.

In [2]:
import os
import asyncio
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from neo4j import GraphDatabase

from neo4j_graphrag.llm import OpenAILLM
from neo4j_graphrag.embeddings import OpenAIEmbeddings
from neo4j_graphrag.experimental.pipeline.kg_builder import SimpleKGPipeline

print("Libraries imported successfully")

Libraries imported successfully


## 3. Configure Environment Variables and Credentials

Credentials are loaded from the local `.env` file in this folder (Neo4j URI/user/password and the OpenAI API key). Update the fallback values below if you are pointing at a different Neo4j instance.

In [3]:
# Load variables from the local .env (if present)
import os

os.chdir('/content/Agentic_KAG_Workshop_DHS_2026/')
load_dotenv()

# ---- Neo4j connection ----
# NEO4J_URI      = os.getenv("NEO4J_URI")
# NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
# NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
# NEO4J_DATABASE = os.getenv("NEO4J_DATABASE")

NEO4J_URI="bolt://100.31.192.102"
NEO4J_USERNAME="neo4j"
NEO4J_PASSWORD="tactics-lieutenants-seam"
NEO4J_DATABASE="neo4j"

# --- OpenAI ---
os.environ.setdefault(
    'OPENAI_API_KEY',
    os.getenv('OPENAI_API_KEY')
)

print("Neo4j URI     :", NEO4J_URI)
print("Neo4j database:", NEO4J_DATABASE)
print("OpenAI key set:", bool(os.getenv("OPENAI_API_KEY")))

Neo4j URI     : bolt://100.31.192.102
Neo4j database: neo4j
OpenAI key set: True


## 4. Connect to Neo4j Database

Create the driver and verify connectivity before running the pipeline. We also define a small `run_query` helper used throughout the notebook.

In [14]:
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
driver.verify_connectivity()
print("Connected to Neo4j")


def run_query(query, params=None):
    """Run a Cypher query and return the results as a list of dicts."""
    with driver.session(database=NEO4J_DATABASE) as session:
        return [record.data() for record in session.run(query, params or {})]

Connected to Neo4j


## 5. Load the PDF Document

Point to `Leave_Policy.pdf` in the AV2026 folder, confirm the file exists, and peek at the first page of extracted text so we can see the unstructured content the pipeline will process.

In [16]:
from pypdf import PdfReader

PDF_PATH = Path("/content/Agentic_KAG_Workshop_DHS_2026/data/Leave_Policy.pdf")
assert PDF_PATH.exists(), f"File not found: {PDF_PATH}"

reader = PdfReader(str(PDF_PATH))
print(f"File: {PDF_PATH.name}")
print(f"Number of pages: {len(reader.pages)}")

File: Leave_Policy.pdf
Number of pages: 7


## 6. Initialize the LLM and Embedder

- The **LLM** (`gpt-4o-mini`) reads each text chunk and extracts entities & relationships as JSON.
- The **embedder** (`text-embedding-3-small`) converts every chunk into a vector, which becomes a property on the `Chunk` nodes of the lexical graph (enabling vector search later).

In [17]:
llm = OpenAILLM(
    model_name="gpt-5-mini",
    model_params={
        "temperature": 0,
        "response_format": {"type": "json_object"},
    },
)

embedder = OpenAIEmbeddings(model="text-embedding-3-small")

print("LLM and embedder configured")

LLM and embedder configured


## 7. Define the Graph Schema

A schema *guides* the LLM so extraction stays consistent and relevant to an HR **leave policy**. We define:

- **Node types** – the domain entities we care about (`Policy`, `LeaveType`, `Employee`, `Eligibility`, `Duration`, `Approver`, `Condition`, `Organization`).
- **Relationship types** – how those entities connect.
- **Patterns** – the allowed `(source) -[REL]-> (target)` triples that constrain extraction.

In [18]:
import json

# ---------------------------------------------------------------------------
# Node types (domain entities relevant to a leave policy)
# ---------------------------------------------------------------------------
node_types = [
    {
        "label": "Policy",
        "properties": [
            {"name": "name", "type": "STRING"},
            {"name": "description", "type": "STRING"},
        ],
    },
    {
        "label": "LeaveType",
        "properties": [
            {"name": "name", "type": "STRING"},          # e.g. Annual, Sick, Maternity
            {"name": "description", "type": "STRING"},
        ],
    },
    {"label": "Employee",     "properties": [{"name": "name", "type": "STRING"}, {"name": "category", "type": "STRING"}]},
    {"label": "Eligibility",  "properties": [{"name": "criteria", "type": "STRING"}]},
    {"label": "Duration",     "properties": [{"name": "value", "type": "STRING"}]},   # e.g. "21 days"
    {"label": "Approver",     "properties": [{"name": "role", "type": "STRING"}]},
    {"label": "Condition",    "properties": [{"name": "description", "type": "STRING"}]},
    {"label": "Organization", "properties": [{"name": "name", "type": "STRING"}]},
]

# ---------------------------------------------------------------------------
# Relationship types
# ---------------------------------------------------------------------------
relationship_types = [
    "DEFINES",           # Policy DEFINES LeaveType
    "APPLIES_TO",        # LeaveType APPLIES_TO Employee
    "HAS_ELIGIBILITY",   # LeaveType HAS_ELIGIBILITY Eligibility
    "HAS_DURATION",      # LeaveType HAS_DURATION Duration
    "APPROVED_BY",       # LeaveType APPROVED_BY Approver
    "SUBJECT_TO",        # LeaveType SUBJECT_TO Condition
    "ISSUED_BY",         # Policy ISSUED_BY Organization
]

# ---------------------------------------------------------------------------
# Allowed patterns  (source -> REL -> target)
# ---------------------------------------------------------------------------
patterns = [
    ("Policy",    "DEFINES",         "LeaveType"),
    ("Policy",    "ISSUED_BY",       "Organization"),
    ("LeaveType", "APPLIES_TO",      "Employee"),
    ("LeaveType", "HAS_ELIGIBILITY", "Eligibility"),
    ("LeaveType", "HAS_DURATION",    "Duration"),
    ("LeaveType", "APPROVED_BY",     "Approver"),
    ("LeaveType", "SUBJECT_TO",      "Condition"),
]

## 8. Build the SimpleKGPipeline

We configure `SimpleKGPipeline` with `from_pdf=True` so it reads the PDF directly, chunks it, embeds each chunk (lexical graph), and runs schema-guided entity extraction (domain graph). `perform_entity_resolution=True` merges duplicate entities that the LLM extracts across chunks.

In [19]:
kg_builder = SimpleKGPipeline(
    llm=llm,
    driver=driver,
    embedder=embedder,
    neo4j_database=NEO4J_DATABASE,
    from_pdf=True,                       # read the PDF directly -> builds the lexical graph
    schema={
        "node_types": node_types,
        "relationship_types": relationship_types,
        "patterns": patterns,
    },
    perform_entity_resolution=True,      # merge duplicate extracted entities
    on_error="IGNORE",
)

print("SimpleKGPipeline configured (from_pdf=True)")

/tmp/ipykernel_3221/1674518680.py:1: DeprecationWarning: `from_pdf` is deprecated and will be removed in a future version; use `from_file` instead.
  kg_builder = SimpleKGPipeline(


SimpleKGPipeline configured (from_pdf=True)


## 9. Run the Pipeline on the PDF

Running the pipeline is asynchronous. In a Jupyter notebook we can simply `await` the coroutine. This step extracts chunks, embeds them, extracts entities/relationships, and writes everything to Neo4j.

> Optional: uncomment the first line to start from a clean database.

In [20]:
# Optional: clear the database for a clean demo
# run_query("MATCH (n) DETACH DELETE n")

result = await kg_builder.run_async(file_path=str(PDF_PATH))
print("Pipeline finished")
print(result)

EmbeddingsGenerationError: Failed to generate embedding with OpenAI: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-proj-*****************_KEY. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}

## 10. Inspect the Lexical Graph

The **lexical graph** records the physical structure of the text:
`(:Document) -[:FROM_DOCUMENT]- (:Chunk) -[:NEXT_CHUNK]-> (:Chunk)`.

Each `Chunk` also carries its `text` and an `embedding` vector. Let's look at the documents and a few chunks.

In [ ]:
# Document + chunk counts
lexical_stats = run_query("""
MATCH (d:Document)
OPTIONAL MATCH (c:Chunk)
RETURN count(DISTINCT d) AS documents, count(DISTINCT c) AS chunks
""")
print("Lexical graph stats:", lexical_stats)

# Preview a few chunks (text + whether an embedding is present)
chunks_preview = run_query("""
MATCH (c:Chunk)
RETURN c.index AS index,
       left(c.text, 160) AS text_preview,
       (c.embedding IS NOT NULL) AS has_embedding
ORDER BY index
LIMIT 5
""")
pd.DataFrame(chunks_preview)

## 11. Inspect the Domain Graph

The **domain graph** holds the semantic entities extracted by the LLM (`LeaveType`, `Eligibility`, `Duration`, ...) and the relationships between them. Every extracted entity is also linked back to the `Chunk` it came from via `FROM_CHUNK`, connecting the domain graph to the lexical graph.

In [ ]:
# Count nodes per domain label (exclude the lexical labels)
domain_labels = [n["label"] for n in node_types]
label_counts = run_query("""
CALL db.labels() YIELD label
CALL {
    WITH label
    MATCH (n) WHERE label IN labels(n)
    RETURN count(n) AS count
}
RETURN label, count
ORDER BY count DESC
""")
print("All node labels & counts:")
display(pd.DataFrame(label_counts))

# Relationship type counts
rel_counts = run_query("""
CALL db.relationshipTypes() YIELD relationshipType
CALL {
    WITH relationshipType
    MATCH ()-[r]->() WHERE type(r) = relationshipType
    RETURN count(r) AS count
}
RETURN relationshipType, count
ORDER BY count DESC
""")
print("Relationship types & counts:")
pd.DataFrame(rel_counts)

In [ ]:
# Show the extracted domain triples (entity -[REL]-> entity)
domain_triples = run_query("""
MATCH (a)-[r]->(b)
WHERE NOT a:Chunk AND NOT a:Document
  AND NOT b:Chunk AND NOT b:Document
RETURN labels(a)[0]  AS source_label, coalesce(a.name, a.role, a.value, a.criteria, a.description) AS source,
       type(r)       AS relationship,
       labels(b)[0]  AS target_label, coalesce(b.name, b.role, b.value, b.criteria, b.description) AS target
LIMIT 50
""")
pd.DataFrame(domain_triples)

## 12. Visualize the Knowledge Graph

We render an interactive HTML view of both graphs using `pyvis`. This shows the **lexical graph** (Document → Chunks) alongside the **domain graph** (entities), plus the `FROM_CHUNK` edges that link an entity back to the chunk it was extracted from.

> You can also explore the full graph visually in **Neo4j Browser / Bloom**.

In [ ]:
%pip install -q pyvis
from pyvis.network import Network
from IPython.display import IFrame

# Pull a subgraph: chunks + the entities they mention + entity-entity relations
viz_rows = run_query("""
MATCH (a)-[r]->(b)
RETURN id(a) AS aid, labels(a)[0] AS alabel,
       coalesce(a.name, a.role, a.value, a.criteria, a.description, 'Chunk ' + toString(a.index)) AS aname,
       type(r) AS rel,
       id(b) AS bid, labels(b)[0] AS blabel,
       coalesce(b.name, b.role, b.value, b.criteria, b.description, 'Chunk ' + toString(b.index)) AS bname
LIMIT 150
""")

color_map = {
    "Document": "#8e44ad", "Chunk": "#95a5a6",
    "Policy": "#e74c3c", "LeaveType": "#2980b9", "Employee": "#27ae60",
    "Eligibility": "#f39c12", "Duration": "#16a085", "Approver": "#d35400",
    "Condition": "#c0392b", "Organization": "#2c3e50",
}

net = Network(height="650px", width="100%", directed=True, notebook=True, cdn_resources="in_line")
seen = set()
for row in viz_rows:
    for nid, label, name in [(row["aid"], row["alabel"], row["aname"]),
                             (row["bid"], row["blabel"], row["bname"])]:
        if nid not in seen:
            net.add_node(nid, label=str(name)[:30], title=label,
                         color=color_map.get(label, "#7f8c8d"))
            seen.add(nid)
    net.add_edge(row["aid"], row["bid"], label=row["rel"])

net.repulsion(node_distance=180, spring_length=200)
net.write_html("leave_policy_kg.html", notebook=False)
print("Graph written to leave_policy_kg.html")
IFrame("leave_policy_kg.html", width="100%", height="670px")

## Summary

Starting from a single **unstructured PDF** (`Leave_Policy.pdf`), Neo4j GraphRAG's `SimpleKGPipeline` automatically produced:

- a **lexical graph** — `Document` and `Chunk` nodes with vector embeddings, capturing the text structure, and
- a **domain graph** — `LeaveType`, `Eligibility`, `Duration`, `Approver`, `Condition` entities and their relationships, capturing the *meaning*.

The two graphs are stitched together through `FROM_CHUNK` edges, giving every extracted fact full **provenance** back to its source text. This foundation supports GraphRAG retrieval that reasons over connected policy knowledge instead of isolated text chunks.

Finally, close the driver to release the connection.

In [ ]:
driver.close()
print("Neo4j driver closed")

## 13. Delete the Graph (Cleanup)

Run the cell below to remove **all nodes and relationships** from the database — useful for resetting between demo runs. This is destructive and cannot be undone.

In [ ]:
# WARNING: this deletes ALL nodes and relationships in the database.
# Re-open a driver in case it was closed in the previous cell.
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

with driver.session(database=NEO4J_DATABASE) as session:
    session.run("MATCH (n) DETACH DELETE n")

# Confirm the graph is empty
remaining = run_query("MATCH (n) RETURN count(n) AS node_count")
print("Graph deleted. Remaining nodes:", remaining[0]["node_count"])

driver.close()
print("Neo4j driver closed")